# TTD Preprocessing (DataFrames Only)

This notebook parses the raw TTD files (same logic as `ttd_processing_step.ipynb`) and builds clean DataFrames **without** reading or writing parquet files.

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings

warnings.filterwarnings('ignore')


# ===============================
# CONFIG
# ===============================

In [2]:

DATA_DIR = Path('/home/abhishekh/abhi/biochirp/database/ttd/')

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Missing data directory: {DATA_DIR}")


# ===============================
# HELPERS
# ===============================

In [3]:


def read_rows(file_path: Path, keywords: list[str]) -> pd.DataFrame:
    # Read only rows containing any keyword and split by tab.
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        lines = file.readlines()
    rows = []
    for line in lines:
        if any(keyword in line for keyword in keywords):
            rows.append(line.strip().split('	'))
    # Pad to 5 columns for consistency
    rows = [row if len(row) == 5 else row + [''] * (5 - len(row)) for row in rows]
    return pd.DataFrame(rows)


def normalize_strings(df: pd.DataFrame) -> pd.DataFrame:
    # Safe string normalization (handles duplicate columns).
    df = df.loc[:, ~df.columns.duplicated()].copy()
    for c in df.select_dtypes(include=["object"]).columns:
        series = df[c]
        if isinstance(series, pd.DataFrame):
            series = series.astype("string").agg(" ".join, axis=1)
        df[c] = series.astype("string").str.strip()
    return df

def drop_missing_required(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    # Drop rows missing required columns.
    keep = [c for c in cols if c in df.columns]
    if not keep:
        return df
    return df.dropna(subset=keep)


def strip_semicolons_if_present(df: pd.DataFrame) -> pd.DataFrame:
    # If any cell contains ';', keep only text before first ';' for all string columns.
    obj_cols = df.select_dtypes(include=['object']).columns
    if len(obj_cols) == 0:
        return df
    has_semicolon = df[obj_cols].astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any()
    if not has_semicolon:
        return df
    df = df.copy()
    df[obj_cols] = df[obj_cols].astype(str).replace({r';.*$': ''}, regex=True).apply(lambda s: s.str.strip())
    return df


def extract_icd11(text: str | None) -> str | None:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None
    s = str(text)
    m = re.search(r'ICD-11:\s*([^\]\);]+)', s)
    if m:
        return m.group(1).strip()
    m2 = re.search(r'\[ICD-11:\s*([^\]\);]+)\]', s)
    if m2:
        return m2.group(1).strip()
    return None


def disease_id_from_icd_or_name(icd11: str | None, name: str | None) -> str | None:
    if icd11 and icd11 != 'N.A.':
        return icd11.strip()
    if name:
        return str(name).strip()
    return None


def resolve_id_priority(df: pd.DataFrame, target_col: str, sources: list[str]) -> pd.DataFrame:
    df = df.copy()
    df[target_col] = np.nan
    for col in sources:
        if col in df.columns:
            # normalize possible "ICD-11: X" strings
            tmp = df[col].astype("string").str.replace(r"^ICD-\\d+:\\s*", "", regex=True).str.strip()
            df[target_col] = df[target_col].fillna(tmp)
    return df



# ===============================
# TARGET–PATHWAY + BIOMARKER–DISEASE
# ===============================

In [4]:


target_pathway_mapping = pd.read_table(DATA_DIR / 'P4-01-Target-KEGGpathway_all.txt', skiprows=16)
target_pathway_mapping.columns = ['target_id', 'pathway_id', 'pathway_name']
target_pathway_mapping = normalize_strings(target_pathway_mapping)
target_pathway_mapping = drop_missing_required(target_pathway_mapping, ['target_id', 'pathway_id',  'pathway_name'])
target_pathway_mapping = target_pathway_mapping[['target_id', 'pathway_id', 'pathway_name']].drop_duplicates()

target_pathway_mapping = strip_semicolons_if_present(target_pathway_mapping)

# Pathway master table (single)
pathway_master_table = (
    target_pathway_mapping[['pathway_id', 'pathway_name']]
    .dropna()
    .drop_duplicates(subset=['pathway_id'])
    .sort_values('pathway_id')
    .reset_index(drop=True)
)

# Target–pathway association (edge table)
target_pathway_association = (
    target_pathway_mapping[['target_id', 'pathway_id']]
    .dropna()
    .drop_duplicates(subset=['target_id', 'pathway_id'])
    .sort_values(['target_id', 'pathway_id'])
    .reset_index(drop=True)
)

# Biomarker–Disease mapping
bio_marker_to_disease_mapping = pd.read_table(DATA_DIR / 'P1-08-Biomarker_disease.txt', skiprows=15)
bio_marker_to_disease_mapping.columns = ['biomarker_id', 'biomarker_name', 'disease_name', 'icd_11_code_for_disease', 'icd_10_code_for_disease', 'icd_9_code_for_disease']
bio_marker_to_disease_mapping = bio_marker_to_disease_mapping[bio_marker_to_disease_mapping['biomarker_id'] != 'BiomarkerID']
bio_marker_to_disease_mapping = normalize_strings(bio_marker_to_disease_mapping)

bio_marker_to_disease_mapping = resolve_id_priority(
    bio_marker_to_disease_mapping,
    "disease_id",
    ["icd_11_code_for_disease", "icd_10_code_for_disease", "icd_9_code_for_disease"]
)

bio_marker_to_disease_mapping['disease_id'] = bio_marker_to_disease_mapping['icd_11_code_for_disease'].apply(extract_icd11)
bio_marker_to_disease_mapping['disease_id'] = bio_marker_to_disease_mapping.apply(
    lambda r: disease_id_from_icd_or_name(r['disease_id'], r['disease_name']), axis=1
)

biomarker_disease_association = bio_marker_to_disease_mapping[['biomarker_id', 'disease_id']].drop_duplicates()

biomarker_disease_association = strip_semicolons_if_present(biomarker_disease_association)

# Biomarker master table (single)
biomarker_master_table = (
    bio_marker_to_disease_mapping[['biomarker_id', 'biomarker_name']]
    .dropna()
    .drop_duplicates(subset=['biomarker_id'])
    .sort_values('biomarker_id')
    .reset_index(drop=True)
)
biomarker_master_table = strip_semicolons_if_present(biomarker_master_table)



In [5]:
# ===============================
# TARGET MASTER TABLE
# ===============================

keywords = [
    'TARGETID', 'FORMERID', 'UNIPROID', 'TARGNAME', 'GENENAME', 'TARGTYPE',
    'FUNCTION', 'PDBSTRUC', 'BIOCLASS', 'ECNUMBER', 'SEQUENCE'
]

target_master_raw = read_rows(DATA_DIR / 'P1-01-TTD_target_download.txt', keywords)
# Remove header noise (same as original)
target_master_raw = target_master_raw.iloc[11:, :3]
target_master_raw.columns = ['target_id', 'attribute', 'value']

target_master_table = target_master_raw.pivot(columns='attribute', index='target_id', values='value').reset_index()

rename_map = {
    'TARGETID': 'target_id',
    'TARGNAME': 'target_name',
    'GENENAME': 'gene_name',
    'TARGTYPE': 'target_evidence_type',
    'BIOCLASS': 'target_class',
    'ECNUMBER': 'ec_number',
    'FORMERID': 'target_former_identifier',
    'FUNCTION': 'target_biological_function',
    'PDBSTRUC': 'pdb_structure',
    'SEQUENCE': 'aa_sequence',
    'UNIPROID': 'target_uniprot_id'
}

target_master_table = target_master_table.rename(columns=rename_map)

keep_cols = ['target_id', 'target_name', 'gene_name']
for col in keep_cols:
    if col not in target_master_table.columns:
        target_master_table[col] = np.nan

target_master_table = target_master_table[keep_cols]
target_master_table = normalize_strings(target_master_table)

# Expand multi-gene symbols
if 'gene_name' in target_master_table.columns:
    target_master_table['gene_name'] = target_master_table['gene_name'].astype('string').str.split(';')
    target_master_table = target_master_table.explode('gene_name')
    target_master_table['gene_name'] = target_master_table['gene_name'].astype('string').str.strip()

# Remove semicolons (if any remain)
target_master_table = strip_semicolons_if_present(target_master_table)

target_master_table = (
    target_master_table[['target_id', 'target_name', 'gene_name']]
    .dropna()
    .drop_duplicates()
    .sort_values(['target_id', 'target_name', 'gene_name'])
    .reset_index(drop=True)
)



# ===============================
# TARGET–DISEASE ASSOCIATION
# ===============================

In [6]:
keywords = ['INDICATI']

target_disease_relation_table = read_rows(DATA_DIR / 'P1-06-Target_disease.txt', keywords)
target_disease_relation_table = target_disease_relation_table.iloc[1:, :]
target_disease_relation_table.columns = ['target_id', 'INDICATI', 'target_disease_approval_status', 'disease_name', 'icd_11_code_for_disease']

target_disease_relation_table = target_disease_relation_table[target_disease_relation_table['target_id'] != 'INDICATI']
target_disease_relation_table = target_disease_relation_table[['target_id', 'disease_name', 'target_disease_approval_status', 'icd_11_code_for_disease']].drop_duplicates()

target_disease_relation_table = normalize_strings(target_disease_relation_table)

_target_icd = target_disease_relation_table['icd_11_code_for_disease'].apply(extract_icd11)
target_disease_relation_table['disease_id'] = target_disease_relation_table.apply(
    lambda r: disease_id_from_icd_or_name(_target_icd.loc[r.name], r['disease_name']), axis=1
)

# Association table
target_disease_association = target_disease_relation_table[['target_id', 'disease_id']].drop_duplicates()
target_disease_association = strip_semicolons_if_present(target_disease_association)


In [7]:
# ===============================
# DRUG MASTER + DRUG–TARGET
# ===============================

keywords = [
    'DRUG__ID', 'TRADNAME', 'DRUGCOMP', 'THERCLAS', 'DRUGTYPE',
    'DRUGINCH', 'DRUGINKE', 'DRUGSMIL', 'COMPCLAS'
]

drug_master_raw = read_rows(DATA_DIR / 'P1-02-TTD_drug_download.txt', keywords)
drug_master_raw = drug_master_raw.iloc[10:, :3]
drug_master_raw.columns = ['drug_id', 'attribute', 'value']

drug_master_table = drug_master_raw.pivot(columns='attribute', index='drug_id', values='value').reset_index()

# Prevent duplicate drug_id columns before rename
if 'DRUG__ID' in drug_master_table.columns and 'drug_id' in drug_master_table.columns:
    drug_master_table = drug_master_table.drop(columns=['DRUG__ID'])

rename_map = {
    'DRUG__ID': 'drug_id',
    'DRUGCOMP': 'drug_components',
    'THERCLAS': 'drug_therapeutic_class',
    'DRUGTYPE': 'drug_type',
    'DRUGINCH': 'drug_inchi',
    'DRUGINKE': 'drug_inchi_key',
    'DRUGSMIL': 'drug_smiles',
    'COMPCLAS': 'drug_composition_class',
    'TRADNAME': 'drug_trade_name'
}

drug_master_table = drug_master_table.rename(columns=rename_map)

# Ensure unique column labels before merge
if drug_master_table.columns.duplicated().any():
    drug_master_table = drug_master_table.loc[:, ~drug_master_table.columns.duplicated()].copy()



# Pull drug_name from DRUGINFO lines
keywords = ['DRUGINFO']
drug_related_to_target = read_rows(DATA_DIR / 'P1-01-TTD_target_download.txt', keywords)
drug_related_to_target.columns = ['target_id', 'DRUGINFO', 'drug_id', 'drug_name', 'highest']
drug_related_to_target = drug_related_to_target.loc[1:, :]
drug_related_to_target = drug_related_to_target.drop(columns=['DRUGINFO'])

drug_master_table_id_name = drug_related_to_target[['drug_id', 'drug_name']].drop_duplicates()

drug_master_table = drug_master_table.merge(drug_master_table_id_name, on='drug_id', how='left')

if 'drug_trade_name' in drug_master_table.columns:
    drug_master_table['drug_name'] = drug_master_table['drug_name'].fillna(drug_master_table['drug_trade_name'])

keep_cols = ['drug_id', 'drug_name']
for col in keep_cols:
    if col not in drug_master_table.columns:
        drug_master_table[col] = np.nan

drug_master_table = drug_master_table[keep_cols]
drug_master_table = normalize_strings(drug_master_table)
drug_master_table = strip_semicolons_if_present(drug_master_table)

drug_master_table = (
    drug_master_table[['drug_id', 'drug_name']]
    .dropna()
    .drop_duplicates()
    .sort_values(['drug_id', 'drug_name'])
    .reset_index(drop=True)
)


# Drug–Target mapping
drug_target_mapping = pd.read_excel(DATA_DIR / 'P1-07-Drug-TargetMapping.xlsx')
rename_map = {
    'TargetID': 'target_id',
    'DrugID': 'drug_id',
    'Highest_status': 'drug_approval_status',
    'MOA': 'drug_mechanism_of_action_on_target'
}
drug_target_mapping = drug_target_mapping.rename(columns=rename_map)
if 'drug_mechanism_of_action_on_target' in drug_target_mapping.columns:
    drug_target_mapping['drug_mechanism_of_action_on_target'] = drug_target_mapping['drug_mechanism_of_action_on_target'].astype('string').str.split(';')
    drug_target_mapping = drug_target_mapping.explode('drug_mechanism_of_action_on_target')
    drug_target_mapping['drug_mechanism_of_action_on_target'] = drug_target_mapping['drug_mechanism_of_action_on_target'].astype('string').str.strip()

drug_target_association = drug_target_mapping[['target_id', 'drug_id', 'drug_mechanism_of_action_on_target']].drop_duplicates()
drug_target_association = strip_semicolons_if_present(drug_target_association)


In [8]:
# ===============================
# DRUG–DISEASE ASSOCIATION
# ===============================

keywords = ['TTDDRUID', 'INDICATI']

drug_disease_table = read_rows(DATA_DIR / 'P1-05-Drug_disease.txt', keywords)

parsed_data = []
current_drug_id = None

for _, row in drug_disease_table.iterrows():
    if row[0] == 'TTDDRUID' and row[1] != 'TTD Drug ID':
        current_drug_id = row[1]
    elif row[0] == 'INDICATI' and current_drug_id is not None:
        parts = [p for p in row.tolist() if p not in [None, '']]
        indication = parts[1] if len(parts) > 1 else None

        disease_entry = None
        icd11 = None
        status = None

        if len(parts) >= 5:
            disease_entry = parts[2]
            icd11 = parts[3]
            status = parts[4]
        elif len(parts) == 4:
            if isinstance(parts[2], str) and parts[2].startswith('ICD-11'):
                disease_entry = indication
                icd11 = parts[2]
                status = parts[3]
            else:
                disease_entry = parts[2]
                status = parts[3]
        elif len(parts) == 3:
            disease_entry = parts[1]
            status = parts[2]

        disease_name = disease_entry if disease_entry else indication

        parsed_data.append({
            'drug_id': current_drug_id,
            'disease_name': disease_name,
            'icd_11_code_for_disease': icd11,
            'approval_status': status
        })

drug_disease_table_wide = pd.DataFrame(parsed_data)

drug_disease_table_wide['disease_id'] = drug_disease_table_wide['icd_11_code_for_disease'].apply(extract_icd11)

drug_disease_table_wide['disease_id'] = drug_disease_table_wide.apply(
    lambda r: disease_id_from_icd_or_name(r['disease_id'], r['disease_name']), axis=1
)

drug_disease_association = drug_disease_table_wide[['drug_id', 'disease_id', 'approval_status']].drop_duplicates()
drug_disease_association = strip_semicolons_if_present(drug_disease_association)
# Enforce no NaN + deduplicate

drug_disease_association = drug_disease_association.dropna().drop_duplicates().reset_index(drop=True)


In [9]:
# ===============================
# DISEASE MASTER TABLE
# ===============================

disease_master_table = pd.concat([
    target_disease_relation_table[['disease_id', 'disease_name']],
    drug_disease_table_wide[['disease_id', 'disease_name']],
    bio_marker_to_disease_mapping[['disease_id', 'disease_name']]
], ignore_index=True)

disease_master_table = disease_master_table[disease_master_table["disease_id"]!="#N/A"]

disease_master_table = (
    disease_master_table
    .dropna()
    .drop_duplicates(subset=['disease_id'])
    .sort_values('disease_id')
    .reset_index(drop=True)
)


In [10]:
print('='*80)
print('Therapeutic Target Database SUMMARY')
print('='*80)

print('📊 MASTER TABLES:')
print(f'  • target_master_table: {len(target_master_table):,} rows')
print(f'    Columns: {list(target_master_table.columns)}')
print(f'• drug_master_table: {len(drug_master_table):,} rows')
print(f'    Columns: {list(drug_master_table.columns)}')
print(f'• disease_master_table: {len(disease_master_table):,} rows')
print(f'    Columns: {list(disease_master_table.columns)}')
print(f'• pathway_master_table: {len(pathway_master_table):,} rows')
print(f'    Columns: {list(pathway_master_table.columns)}')
print(f'• biomarker_master_table: {len(biomarker_master_table):,} rows')
print(f'    Columns: {list(biomarker_master_table.columns)}')

print('🔗 EDGE TABLES:')
print(f'  • target_pathway_association: {len(target_pathway_association):,} rows')
print(f'    Columns: {list(target_pathway_association.columns)}')
print(f'• biomarker_disease_association: {len(biomarker_disease_association):,} rows')
print(f'    Columns: {list(biomarker_disease_association.columns)}')
print(f'• target_disease_association: {len(target_disease_association):,} rows')
print(f'    Columns: {list(target_disease_association.columns)}')
print(f'• drug_target_association: {len(drug_target_association):,} rows')
print(f'    Columns: {list(drug_target_association.columns)}')
print(f'• drug_disease_association: {len(drug_disease_association):,} rows')
print(f'    Columns: {list(drug_disease_association.columns)}')

print(f"Number of unique Mechanism of action: {len(set(drug_target_association['drug_mechanism_of_action_on_target']))}")
print(f"Number of unique approval status: {len(set(drug_disease_association['approval_status']))}")
print(f"Number of unique target name: {len(set(target_master_table['target_name']))}")
print(f"Number of unique biomarker name: {len(set(biomarker_master_table['biomarker_name']))}")


print(f"Number of unique drug name: {len(set(drug_master_table['drug_name']))}")
print(f"Number of unique gene name: {len(set(target_master_table['gene_name']))}")
print(f"Number of unique disease name: {len(set(disease_master_table['disease_name']))}")
print(f"Number of unique pathway name: {len(set(pathway_master_table['pathway_name']))}")

print('' + '='*80)
print('✅ TTD PREPROCESSING COMPLETE!')
print('='*80)

Therapeutic Target Database SUMMARY
📊 MASTER TABLES:
  • target_master_table: 3,859 rows
    Columns: ['target_id', 'target_name', 'gene_name']
• drug_master_table: 32,660 rows
    Columns: ['drug_id', 'drug_name']
• disease_master_table: 1,660 rows
    Columns: ['disease_id', 'disease_name']
• pathway_master_table: 379 rows
    Columns: ['pathway_id', 'pathway_name']
• biomarker_master_table: 2,512 rows
    Columns: ['biomarker_id', 'biomarker_name']
🔗 EDGE TABLES:
  • target_pathway_association: 8,531 rows
    Columns: ['target_id', 'pathway_id']
• biomarker_disease_association: 2,512 rows
    Columns: ['biomarker_id', 'disease_id']
• target_disease_association: 10,818 rows
    Columns: ['target_id', 'disease_id']
• drug_target_association: 45,468 rows
    Columns: ['target_id', 'drug_id', 'drug_mechanism_of_action_on_target']
• drug_disease_association: 30,077 rows
    Columns: ['drug_id', 'disease_id', 'approval_status']
Number of unique Mechanism of action: 48
Number of unique app

In [11]:
# biomarker_master_table[biomarker_master_table["biomarker_name"]=="1-acylglycerol-3-phosphate O-acyltransferase PNPLA3 (PNPLA3)"]

In [12]:
# same_name_diff_id = biomarker_master_table.groupby("biomarker_name")["biomarker_id"].nunique()
# same_name_diff_id

In [13]:
# biomarker_master_table.drop_duplicates(["biomarker_name"])

In [14]:
# drug_disease_association["approval_status"].value_counts()

In [15]:
# len(set(drug_target_association["drug_mechanism_of_action_on_target"]))

In [16]:
# ===============================
# DROP ROWS WITH '' OR '.' IN ANY CELL
# ===============================

def drop_empty_or_dot_rows(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.apply(lambda col: col.astype(str).isin(["", "."]), axis=0).any(axis=1)
    return df.loc[~mask].reset_index(drop=True)

# Apply to final tables before NaN/duplicate checks

target_master_table = drop_empty_or_dot_rows(target_master_table)
drug_master_table = drop_empty_or_dot_rows(drug_master_table)
disease_master_table = drop_empty_or_dot_rows(disease_master_table)
pathway_master_table = drop_empty_or_dot_rows(pathway_master_table)
biomarker_master_table = drop_empty_or_dot_rows(biomarker_master_table)

target_pathway_association = drop_empty_or_dot_rows(target_pathway_association)
biomarker_disease_association = drop_empty_or_dot_rows(biomarker_disease_association)
target_disease_association = drop_empty_or_dot_rows(target_disease_association)
drug_target_association = drop_empty_or_dot_rows(drug_target_association)
drug_disease_association = drop_empty_or_dot_rows(drug_disease_association)


In [17]:
tables = {
    "target_master_table": target_master_table,
    "drug_master_table": drug_master_table,
    "disease_master_table": disease_master_table,
    "pathway_master_table": pathway_master_table,
    "biomarker_master_table": biomarker_master_table,
    "target_pathway_association": target_pathway_association,
    "biomarker_disease_association": biomarker_disease_association,
    "target_disease_association": target_disease_association,
    "drug_target_association": drug_target_association,
    "drug_disease_association": drug_disease_association,
}

for name, df in tables.items():
    if df.isna().any().any():
        nan_cols = df.columns[df.isna().any()].tolist()
        raise ValueError(f"{name} contains NaN in columns: {nan_cols}")

print("✅ No NaN values found in final tables.")

✅ No NaN values found in final tables.


In [18]:
# biomarker_master_table

In [19]:
len(set(target_master_table["gene_name"]))

3524

In [20]:
# ===============================
# SAVE PARQUET OUTPUTS (SAFE)
# ===============================

from pathlib import Path
import polars as pl

OUT_DIR = Path('/home/abhishekh/abhi/biochirp/database/ttd')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def write_parquet_safe(df, path):
    pl.from_pandas(df).write_parquet(
        path,
        compression="zstd",
        statistics=True,
        row_group_size=100_000
    )

# ===============================
# MASTER TABLES
# ===============================

write_parquet_safe(
    target_master_table,
    OUT_DIR / 'P1-01-TTD_target_download.parquet'
)

write_parquet_safe(
    drug_master_table,
    OUT_DIR / 'P1-02-TTD_drug_download.parquet'
)

write_parquet_safe(
    disease_master_table,
    OUT_DIR / 'disease_master_table_ttd.parquet'
)

write_parquet_safe(
    pathway_master_table,
    OUT_DIR / 'pathway_master_table_ttd.parquet'
)

write_parquet_safe(
    biomarker_master_table,
    OUT_DIR / 'biomarker_master_table_ttd.parquet'
)

# ===============================
# ASSOCIATION TABLES
# ===============================

write_parquet_safe(
    target_pathway_association,
    OUT_DIR / 'P4-01-Target-KEGGpathway_all.parquet'
)

write_parquet_safe(
    biomarker_disease_association,
    OUT_DIR / 'P1-08-Biomarker_disease.parquet'
)

write_parquet_safe(
    target_disease_association,
    OUT_DIR / 'P1-06-Target_disease.parquet'
)

write_parquet_safe(
    drug_target_association,
    OUT_DIR / 'P1-07-Drug-TargetMapping.parquet'
)

write_parquet_safe(
    drug_disease_association,
    OUT_DIR / 'P1-05-Drug_disease.parquet'
)

print("✅ Saved Polars-safe parquet files to", OUT_DIR)


✅ Saved Polars-safe parquet files to /home/abhishekh/abhi/biochirp/database/ttd


In [21]:
# biomarker_master_table